# **Examining Occupational Stereotypes (Ngrams)**
## **Goal**: Test relationships between occupational demographics and professions' feminization–masculinization.

### **Setup**
#### Imports

In [10]:
%matplotlib inline

%load_ext autoreload
%autoreload 2

import math
import os
import getpass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

from ngramprep.common.w2v_model import W2VModel
from analyze.cosine_similarity_time_series import cosine_similarity_over_years, plot_nearest_neighbors
from analyze.weat_time_series import compute_weat_over_years
from analyze.dimension_projection_time_series import compute_projection_over_years, compute_baseline_set
from analyze.pca_dimension_time_series import compute_pca_dimension_over_years
from analyze.semantic_drift import track_local_semantic_change, track_global_semantic_change, track_directional_drift
from analyze.average_relatedness_by_year import track_word_relatedness
from analyze.bls_utils import calculate_women_percentage, scrape_bls_professions_csv_batch
from analyze.ipums_utils import (
    aggregate_ipums_professions_csv,
    aggregate_ipums_professions_csv_batch,
    fetch_ipums_microdata_cps,
)
from ipumspy import IpumsApiClient, MicrodataExtract

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


#### Specify model directory

In [2]:
model_path =  '/scratch/edk202/NLP_models/Google_Books/20200217/eng/5gram_files/models_final/norm_and_align'

### **Generate gender projection**
#### Use the `compute_meandiff_dimension` method to create a masculinization dimension

In [ ]:
year = 2019
model = W2VModel(f'{model_path}/w2v_y{year}_wbnone_vs300_w004_mc001_sg1_e010.kv')

gender_contrasts = [
    ('she', 'he'),
    ('her', 'his'),
    ('herself', 'himself'),
    ('woman', 'man'),
    ('women', 'men'),
    ('girl', 'boy'),
    ('girls', 'boys'),
    ('mother', 'father'),
    ('mothers', 'fathers'),
    ('daughter', 'son'),
    ('daughters', 'sons'),
    ('sister', 'brother'),
    ('sisters', 'brothers')
]

meandiff_result = model.compute_meandiff_dimension(
    token_contrasts=gender_contrasts
)

#### Print various words' projections on the masculinization dimension using the `print_word_projections` method

In [ ]:
test_words = [
    # Clear anchors
    'husband', 'wife',
    'king', 'queen',

    # Historically masculine-stereotyped professions
    'engineer', 'physicist', 'soldier', 'pilot',
    'firefighter', 'mechanic', 'carpenter',
    'programmer', 'CEO', 'president',
    'construction_worker', 'plumber',

    # Historically feminine-stereotyped professions
    'nurse', 'teacher', 'secretary', 'receptionist',
    'librarian', 'babysitter', 'housekeeper',
    'caregiver', 'assistant', 'hairdresser',

    # Leadership / status roles (interesting test cases)
    'leader', 'manager', 'chairperson',
    'lawyer', 'doctor', 'scientist',
    'professor', 'judge',

    # Trait terms (useful for validating axis content)
    'strong', 'assertive', 'dominant',
    'competitive', 'logical', 'independent',
    'gentle', 'nurturing', 'empathetic',
    'emotional', 'supportive', 'caring'
]

projections = W2VModel.print_word_projections(model, meandiff_result['dimension'], test_words)

### **Examine time-series**
#### Auto-generate a baseline trend using `compute_baseline_set` function

In [ ]:
baseline_source = compute_baseline_set(
    model_dir=model_path,
    contrast_pairs=gender_contrasts,
    start_year=1900,
    end_year=2019,
    year_step=1,
    method='meandiff',
    exclusion_pattern=r"(man|men|woman|women|girl|boy|girls|boys|mother|father|mothers|fathers|daughter|son|daughters|sons|sister|brother)",
    eps_mean=0.08,
    eps_trend=0.010,
    eps_sigma=0.15,
    min_years=5,
    agg="mean",
    corr_n_permutations=0,
    verbose=True,
    plot_baseline=True
)

baseline_source = baseline_source["baseline"]

#### Altneratively, specify a set of baseline words

In [ ]:
baseline_source = [
    'table', 'chair', 'window', 'door', 'wall', 'floor', 'roof', 'tree', 'leaf', 'branch', 'root', 'flower',
    'grass', 'water', 'stone', 'rock', 'sand', 'soil', 'clay', 'book', 'page', 'paper', 'pen', 'ink', 'cup',
    'plate', 'bowl', 'spoon', 'knife', 'fork', 'bread', 'cheese', 'butter', 'salt', 'sugar', 'road', 'path',
    'bridge', 'gate', 'fence', 'box', 'bag', 'basket', 'bottle', 'jar', 'horse', 'cow', 'sheep', 'pig',
    'chicken', 'dog', 'cat', 'sun', 'moon', 'star', 'cloud', 'rain', 'wind', 'river', 'lake', 'mountain',
    'hill', 'valley', 'wood', 'metal', 'iron', 'gold', 'silver', 'wheel', 'rope', 'chain', 'nail', 'hammer'
]

#### Specify the set of occupations from Caliskan et al. (2017)

In [ ]:
targets = [
    "technician", "accountant", "supervisor", "engineer", "worker", "educator", "clerk", "counselor",
    "inspector", "mechanic", "manager", "therapist", "administrator", "salesperson", "receptionist", "librarian",
    "advisor", "pharmacist", "janitor", "psychologist", "physician", "carpenter", "nurse", "investigator",
    "bartender", "specialist", "electrician", "officer", "pathologist", "teacher", "lawyer", "planner",
    "practitioner", "plumber", "instructor", "surgeon", "veterinarian", "paramedic", "examiner", "chemist",
    "machinist", "appraiser", "nutritionist", "architect", "hairdresser", "baker", "programmer", "paralegal",
    "hygienist", "scientist"
]

#### Compute and plot masculinization time-series

In [ ]:
result = compute_projection_over_years(
    model_dir=model_path,
    token_contrasts=gender_contrasts,
    test_words=targets,
    start_year=1900,
    end_year=2019,
    year_step=1,
    method='meandiff',
    ensure_sign_positive=True,
    smooth=True,
    sigma=2,
    verbose=False,
    baseline_source=baseline_source,
    plot_corrected_if_baseline=True,
    plot=True,
 )

In [ ]:
# Correlation analysis with NaN handling and baseline correction option
year_of_interest = 2015
bls_csv_path = '/scratch/edk202/lexichron/bls_scraped/professionsBLS_2015.csv'
use_baseline_corrected = True  # Set to True to use baseline-corrected projections if available

# Recompute BLS percentages for all targets
bls_percentages = {}
for profession in targets:
    try:
        pct = calculate_women_percentage(bls_csv_path, profession)
        bls_percentages[profession] = pct
    except (ZeroDivisionError, ValueError):
        pass

# Determine which projection data to use
if use_baseline_corrected and result.get('baseline_applied') and not result['projections_corrected'].empty:
    proj_year = result['projections_corrected'].loc[year_of_interest]
    projection_type = "Baseline-corrected"
    print(f"Using baseline-corrected projections")
else:
    proj_year = result['projections'].loc[year_of_interest]
    projection_type = "Raw"
    print(f"Using raw projections")

# Create common_profs: professions with both BLS data and projections for this year
common_profs = [p for p in targets if p in bls_percentages and p in proj_year.index]

# Filter to professions with valid (non-NaN) projection values
valid_profs = [p for p in common_profs if not pd.isna(proj_year[p])]

print(f"Valid professions (non-NaN projections): {len(valid_profs)} out of {len(common_profs)}")
print(f"Filtered out {len(common_profs) - len(valid_profs)} professions with NaN projections")

# Extract values for valid professions
bls_vals = np.array([bls_percentages[p] for p in valid_profs])
proj_vals = np.array([proj_year[p] for p in valid_profs])

# Compute correlation
if len(valid_profs) >= 3:
    corr, p_value = pearsonr(bls_vals, proj_vals)
    
    print(f"\n=== Correlation Results ({projection_type}) ===")
    print(f"Pearson r = {corr:.4f}")
    print(f"p-value = {p_value:.6f}")
    print(f"n = {len(valid_profs)} professions")
    
    # Create scatter plot
    plt.figure(figsize=(10, 7))
    plt.scatter(bls_vals, proj_vals, alpha=0.6, s=120, edgecolors='black', linewidth=0.5)
    
    # Add profession labels
    for i, prof in enumerate(valid_profs):
        plt.annotate(prof, (bls_vals[i], proj_vals[i]), 
                    fontsize=9, alpha=0.8, ha='center')
    
    # Add regression line
    z = np.polyfit(bls_vals, proj_vals, 1)
    p = np.poly1d(z)
    x_line = np.linspace(bls_vals.min(), bls_vals.max(), 100)
    plt.plot(x_line, p(x_line), "r--", alpha=0.8, linewidth=2, label=f'Linear fit')
    
    plt.xlabel("BLS Women Percentage (2015)", fontsize=12, fontweight='bold')
    plt.ylabel(f"Gender Projection Score ({year_of_interest}) - {projection_type}", fontsize=12, fontweight='bold')
    plt.title(f"Word Embeddings vs. Real-World Gender Demographics\nPearson r = {corr:.4f}, p = {p_value:.6f}", 
              fontsize=13, fontweight='bold')
    plt.grid(True, alpha=0.3, linestyle='--')
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.show()
    
    # Print interpretation
    if p_value < 0.05:
        print(f"\n✓ Significant correlation (p < 0.05)")
    else:
        print(f"\n✗ Not statistically significant (p >= 0.05)")
else:
    print(f"ERROR: Not enough valid professions ({len(valid_profs)} found)")

#### Make file list for BLS data download

In [ ]:
# Regression guard: exact label-column matching in calculate_women_percentage
bls_root = Path("/scratch/edk202/lexichron/bls_scraped")
found_csvs = sorted(bls_root.glob("professionsBLS*.csv"))
assert found_csvs, "No professionsBLS*.csv files found under /scratch/edk202/lexichron/bls_scraped"

csv_path = str(found_csvs[0])
print("Using CSV:", csv_path)

# Positive controls: should resolve to finite percentages
positive_terms = ["engineer", "nurse", "manager"]
for term in positive_terms:
    pct = calculate_women_percentage(csv_path, term)
    assert math.isfinite(pct), f"Expected finite pct for '{term}', got {pct}"
    assert 0.0 <= pct <= 1.0, f"Expected pct in [0,1] for '{term}', got {pct}"

# Negative controls: should NOT match exact label columns
negative_terms = ["and", "man"]
for term in negative_terms:
    try:
        _ = calculate_women_percentage(csv_path, term)
        raise AssertionError(f"Expected no exact label match for '{term}', but got a value")
    except ZeroDivisionError:
        pass

print("✅ Exact label matching regression check passed")

### **IPUMS workflow**
#### Convert IPUMS microdata to BLS-compatible yearly profession CSVs

In [ ]:
# IPUMS extract configuration (edit paths/column names to match your extract)
ipums_extract_file = '/scratch/edk202/lexichron/notebooks/ipums_extract.csv'
ipums_occ_map_file = '/scratch/edk202/lexichron/notebooks/ipums_occ1990_map.csv'  # columns: code,label
ipums_output_dir = '/scratch/edk202/lexichron/notebooks/ipums_scraped'

# Export one BLS-compatible CSV per year
ipums_runs_df = aggregate_ipums_professions_csv_batch(
    extract_file=ipums_extract_file,
    output_dir=ipums_output_dir,
    output_basename='professionsIPUMS.csv',
    occupation_code_col='OCC1990',
    occupation_label_col=None,
    occupation_map_file=ipums_occ_map_file,
    year_col='YEAR',
    sex_col='SEX',
    race_col='RACE',
    hispanic_col='HISPAN',
    weight_col='PERWT',
    female_codes=(2,),
    black_codes=(2,),
    asian_codes=(4, 5, 6),
    non_hispanic_codes=(0, 9),
    min_total_employed=50,
    continue_on_error=True,
    inject_year_in_filename=True,
    years=None,  # set e.g. [1990, 2000, 2010] to limit
 )

print(f"Completed: {(ipums_runs_df['status'] == 'ok').sum()} succeeded, {(ipums_runs_df['status'] == 'failed').sum()} failed")
ipums_runs_df.head(10)

In [ ]:
# Smoke test: IPUMS -> BLS-compatible CSV -> exact profession lookup
smoke_extract = '/scratch/edk202/lexichron/notebooks/ipums_extract_smoketest.csv'
smoke_map = '/scratch/edk202/lexichron/notebooks/ipums_occ1990_map_smoketest.csv'
smoke_out_dir = Path('/scratch/edk202/lexichron/notebooks/ipums_smoketest_out')
smoke_out_dir.mkdir(parents=True, exist_ok=True)

runs = aggregate_ipums_professions_csv_batch(
    extract_file=smoke_extract,
    output_dir=str(smoke_out_dir),
    years=[2010, 2011],
    output_basename='professionsIPUMS_smoke.csv',
    occupation_code_col='OCC1990',
    occupation_label_col=None,
    occupation_map_file=smoke_map,
    year_col='YEAR',
    sex_col='SEX',
    race_col='RACE',
    hispanic_col='HISPAN',
    weight_col='PERWT',
    female_codes=(2,),
    black_codes=(2,),
    asian_codes=(4, 5, 6),
    non_hispanic_codes=(0, 9),
    min_total_employed=1,
    continue_on_error=False,
    inject_year_in_filename=True,
 )

assert (runs['status'] == 'ok').all(), f"Batch failed:\n{runs}"

csv_2010 = runs.loc[runs['year'] == 2010, 'output_csv'].iloc[0]
csv_2011 = runs.loc[runs['year'] == 2011, 'output_csv'].iloc[0]

eng_2010 = calculate_women_percentage(csv_2010, 'engineer')
nurse_2010 = calculate_women_percentage(csv_2010, 'nurse')

# 2010 expected values from synthetic data:
# engineers = women weight (120+80) / total (100+120+80) = 200/300 = 0.6667
# nurses = women weight 200 / total (200+50) = 200/250 = 0.8
assert math.isclose(eng_2010, 2/3, rel_tol=1e-3), f"Unexpected engineer pct: {eng_2010}"
assert math.isclose(nurse_2010, 0.8, rel_tol=1e-3), f"Unexpected nurse pct: {nurse_2010}"

print('✅ IPUMS smoke test passed')
runs

#### Live IPUMS CPS ASEC download
Fetches the most recent CPS ASEC extract with harmonized occupation codes (`OCC2010`) and person weights (`ASECWT`).
Auto-discovers the latest ASEC sample, submits/waits/downloads, and decompresses `.gz` files in place.

In [ ]:
result = fetch_ipums_microdata_cps(
    api_key='59cba10d8a5da536fc06b59d0d34a8b208154b9684e7d4fadbe0a339',
    years=range(1968, 2026),
    download_dir='/scratch/edk202/lexichron/ipums_api_downloads',
)

print(f"\nExtract {result['extract_id']}: {result['status']}")
for f in result['files']:
    print(f'  {Path(f).name}  ({Path(f).stat().st_size / 1024:.0f} KB)')

Discovering available CPS ASEC samples...
  Resolved years [1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971, 1972, 1973, 1974, 1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025] -> samples ['cps1962_03s', 'cps1963_03s', 'cps1964_03s', 'cps1965_03s', 'cps1966_03s', 'cps1967_03s', 'cps1968_03s', 'cps1969_03s', 'cps1970_03s', 'cps1971_03s', 'cps1972_03s', 'cps1973_03s', 'cps1974_03s', 'cps1975_03s', 'cps1976_03s', 'cps1977_03s', 'cps1978_03s', 'cps1979_03s', 'cps1980_03s', 'cps1981_03s', 'cps1982_03s', 'cps1983_03s', 'cps1984_03s', 'cps1985_03s', 'cps1986_03s', 'cps1987_03s', 'cps1988_03s', 'cps1989_03s', 'cps1990_03s', 'cps1991_03s', 'cps1992_03s', 'cps1993_03s', 'cps1994_03s', 'cps1995_03s', 'cps1996_03s', 'cps1997_03s', 'cps